In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

training_dataset = pd.read_csv('../data/raw/train.csv')
testing_dataset = pd.read_csv('../data/raw/test.csv')

train_dimensions = training_dataset.shape
test_dimensions = testing_dataset.shape

print("Train shape:", train_dimensions)
print("Test shape:", test_dimensions)

print("\nFirst 5 rows:")
first_five_rows = training_dataset.head()
print(first_five_rows)

# Phase A — Data එක තේරුම් ගැනීම

### 1. Column Types

In [ ]:
print("=== Column Data Types ===")
column_data_types = training_dataset.dtypes
print(column_data_types)

print("\n=== Numerical vs Categorical Columns ===")
numerical_columns = training_dataset.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = training_dataset.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Numerical ({len(numerical_columns)}):", numerical_columns)
print(f"Categorical ({len(categorical_columns)}):", categorical_columns)

### 2. Cardinality

In [ ]:
print("=== Column Cardinality ===")
column_cardinality = training_dataset.nunique().sort_values(ascending=False)
print(column_cardinality)

print("\n=== Cardinality Ratio (Unique / Total Rows) ===")
total_train_records = len(training_dataset)
cardinality_ratio = (column_cardinality / total_train_records * 100).round(3)
print(cardinality_ratio)

# Phase B — Data Cleaning

### 3. Duplicates

In [ ]:
print("=== Duplicate Rows (Train) ===")
duplicate_row_count = training_dataset.duplicated().sum()
duplicate_row_percentage = (duplicate_row_count / len(training_dataset)) * 100
print(f"Total duplicate rows: {duplicate_row_count:,} ({duplicate_row_percentage:.3f}%)")

print("\n=== Duplicate Rows (Test) ===")
test_duplicate_row_count = testing_dataset.duplicated().sum()
print(f"Total duplicate rows: {test_duplicate_row_count:,}")

### 4. Missing Data

In [ ]:
print("=== Missing Values (Train) ===")
missing_values_per_column = training_dataset.isnull().sum()
print(missing_values_per_column)

print("\n=== Missing Values (Test) ===")
test_missing_values_per_column = testing_dataset.isnull().sum()
print(test_missing_values_per_column)

print("\n=== Missing Value Percentage (Train) ===")
missing_value_percentage = (missing_values_per_column / len(training_dataset) * 100).round(3)
print(missing_value_percentage)

# Phase C — Univariate Analysis

### 5. Descriptive Statistics

In [ ]:
print("=== Descriptive Statistics (Numerical) ===")
dataset_summary_statistics = training_dataset.describe()
print(dataset_summary_statistics)

print("\n=== Descriptive Statistics (Categorical) ===")
categorical_summary_statistics = training_dataset.describe(include=['object', 'str'])
print(categorical_summary_statistics)


### 6. Distribution & Shapes

In [ ]:
# Constants
CLASS_LABELS = ['GALAXY', 'QSO', 'STAR']
CLASS_COLOR_PALETTE = ['#4C72B0', '#DD8452', '#55A868']

# Figure Setup
figure, plot_axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Class distribution
class_counts = training_dataset['class'].value_counts()
class_counts.plot(kind='bar', ax=plot_axes[0], color=CLASS_COLOR_PALETTE)
plot_axes[0].set_title('Class Distribution')
plot_axes[0].set_xlabel('Class')
plot_axes[0].set_ylabel('Count')
plot_axes[0].tick_params(axis='x', rotation=0)

# 2. Spectral type distribution
spectral_type_counts = training_dataset['spectral_type'].value_counts()
spectral_type_counts.plot(kind='bar', ax=plot_axes[1])
plot_axes[1].set_title('Spectral Type Distribution')
plot_axes[1].tick_params(axis='x', rotation=45)

# 3. Galaxy population
galaxy_population_counts = training_dataset['galaxy_population'].value_counts()
galaxy_population_counts.plot(kind='bar', ax=plot_axes[2], color=['#c44e52', '#8172b2', '#937860'])
plot_axes[2].set_title('Galaxy Population Distribution')
plot_axes[2].tick_params(axis='x', rotation=0)

# Output Management
plt.tight_layout()
plt.savefig('../eda_categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Constants
CLASS_LABELS = ['GALAXY', 'QSO', 'STAR']
CLASS_COLOR_PALETTE = ['#4C72B0', '#DD8452', '#55A868']
VIOLIN_POSITIONS = [1, 2, 3]
KDE_X_LIMITS = (-1, 8)

# Figure Setup
figure, plot_axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Violin plot
redshift_data_per_class = [
    training_dataset[training_dataset['class'] == class_label]['redshift'].values
    for class_label in CLASS_LABELS
]

violin_parts = plot_axes[0].violinplot(
    redshift_data_per_class,
    positions=VIOLIN_POSITIONS,
    showmedians=True
)

for violin_body, class_color in zip(violin_parts['bodies'], CLASS_COLOR_PALETTE):
    violin_body.set_facecolor(class_color)
    violin_body.set_alpha(0.7)

plot_axes[0].set_xticks(VIOLIN_POSITIONS)
plot_axes[0].set_xticklabels(CLASS_LABELS)
plot_axes[0].set_title('Redshift Distribution (Violin)')
plot_axes[0].set_ylabel('Redshift')

# 2. KDE plot
for class_label, class_color in zip(CLASS_LABELS, CLASS_COLOR_PALETTE):
    class_subset = training_dataset[training_dataset['class'] == class_label]
    redshift_values = class_subset['redshift']
    redshift_values.plot.kde(ax=plot_axes[1], label=class_label, color=class_color, linewidth=2)

plot_axes[1].set_title('Redshift KDE by Class')
plot_axes[1].set_xlabel('Redshift')
plot_axes[1].legend()
plot_axes[1].set_xlim(KDE_X_LIMITS)

# Output Management
plt.tight_layout()
plt.savefig('../eda_redshift_shape.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Constants
CLASS_LABELS = ['GALAXY', 'QSO', 'STAR']
CLASS_COLOR_PALETTE = ['#4C72B0', '#DD8452', '#55A868']
COLOR_INDEX_COLUMNS = ['u_g', 'g_r', 'r_i', 'i_z']
KDE_X_LIMITS = (-3, 8)

# Feature Engineering
train_features_extended = training_dataset.copy()
train_features_extended['u_g'] = train_features_extended['u'] - train_features_extended['g']
train_features_extended['g_r'] = train_features_extended['g'] - train_features_extended['r']
train_features_extended['r_i'] = train_features_extended['r'] - train_features_extended['i']
train_features_extended['i_z'] = train_features_extended['i'] - train_features_extended['z']

# Figure Setup
figure, subplot_axes = plt.subplots(2, 2, figsize=(14, 10))
flattened_axes = subplot_axes.flatten()

# Plotting KDE for each Color Index
for axis_index, column_name in enumerate(COLOR_INDEX_COLUMNS):
    for class_label, class_color in zip(CLASS_LABELS, CLASS_COLOR_PALETTE):
        class_subset = train_features_extended[train_features_extended['class'] == class_label]
        color_index_values = class_subset[column_name]
        color_index_values.plot.kde(
            ax=flattened_axes[axis_index],
            label=class_label,
            color=class_color,
            linewidth=2
        )

    flattened_axes[axis_index].set_title(f'Color Index: {column_name}')
    flattened_axes[axis_index].set_xlabel(column_name)
    flattened_axes[axis_index].legend()
    flattened_axes[axis_index].set_xlim(KDE_X_LIMITS)

# Output Management
plt.suptitle('Astronomical Color Indices by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../eda_color_indices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Constants
CLASS_LABELS = ['GALAXY', 'QSO', 'STAR']
CLASS_COLOR_PALETTE = ['#4C72B0', '#DD8452', '#55A868']
PHOTOMETRIC_FEATURES = ['u', 'g', 'r', 'i', 'z', 'redshift']
COLUMNS_PER_ROW = 3

# Figure Setup
figure, subplot_axes = plt.subplots(2, 3, figsize=(18, 10))

# Plotting KDE for each Photometric Feature
for feature_index, feature_name in enumerate(PHOTOMETRIC_FEATURES):
    row_index = feature_index // COLUMNS_PER_ROW
    column_index = feature_index % COLUMNS_PER_ROW
    current_axis = subplot_axes[row_index][column_index]

    for class_label, class_color in zip(CLASS_LABELS, CLASS_COLOR_PALETTE):
        class_subset = train_features_extended[train_features_extended['class'] == class_label]
        feature_values = class_subset[feature_name]
        feature_values.plot.kde(
            ax=current_axis,
            label=class_label,
            color=class_color,
            linewidth=2
        )

    current_axis.set_title(f'{feature_name} Distribution by Class')
    current_axis.set_xlabel(feature_name)
    current_axis.legend()

# Output Management
plt.suptitle('Photometric Band Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../eda_bands.png', dpi=150, bbox_inches='tight')
plt.show()

### 7. Outliers

In [ ]:
# Constants
NUMERICAL_FEATURES = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
IQR_MULTIPLIER = 1.5

# Figure Setup
figure, plot_axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Box plot
training_dataset.boxplot(column='redshift', by='class', ax=plot_axes[0])
plot_axes[0].set_title('Redshift by Class (Boxplot)')
plot_axes[0].set_xlabel('Class')
plot_axes[0].set_ylabel('Redshift')
plt.sca(plot_axes[0])
plt.xticks(rotation=0)

# 2. Numerical feature boxplots
training_dataset[NUMERICAL_FEATURES].boxplot(ax=plot_axes[1], rot=45)
plot_axes[1].set_title('Numerical Feature Boxplots')

# Output Management
plt.suptitle('')
plt.tight_layout()
plt.savefig('../eda_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== IQR-Based Outlier Counts ===")
outlier_summary_rows = []
for feature_name in NUMERICAL_FEATURES:
    first_quartile = training_dataset[feature_name].quantile(0.25)
    third_quartile = training_dataset[feature_name].quantile(0.75)
    interquartile_range = third_quartile - first_quartile
    lower_bound = first_quartile - IQR_MULTIPLIER * interquartile_range
    upper_bound = third_quartile + IQR_MULTIPLIER * interquartile_range
    outlier_count = training_dataset[
        (training_dataset[feature_name] < lower_bound) | (training_dataset[feature_name] > upper_bound)
    ].shape[0]
    outlier_percentage = (outlier_count / len(training_dataset)) * 100
    outlier_summary_rows.append({
        'feature': feature_name,
        'lower_bound': round(lower_bound, 3),
        'upper_bound': round(upper_bound, 3),
        'outlier_count': outlier_count,
        'outlier_percentage': round(outlier_percentage, 3)
    })

outlier_summary_dataframe = pd.DataFrame(outlier_summary_rows)
print(outlier_summary_dataframe)

# Phase D — Target Column Check

### 8. Class Imbalance

In [ ]:
print("=== Class Distribution ===")
class_distribution_counts = training_dataset['class'].value_counts()
print(class_distribution_counts)

print("\n=== Class Distribution (%) ===")
total_train_records = len(training_dataset)
for class_label, record_count in class_distribution_counts.items():
    class_percentage = (record_count / total_train_records) * 100
    print(f"   {class_label:<8}: {record_count:>7,} ({class_percentage:.1f}%)")

# Constants
CLASS_LABELS = ['GALAXY', 'QSO', 'STAR']
CLASS_COLOR_PALETTE = ['#4C72B0', '#DD8452', '#55A868']

# Figure Setup
figure, plot_axis = plt.subplots(figsize=(6, 5))
class_distribution_counts.plot(kind='bar', ax=plot_axis, color=CLASS_COLOR_PALETTE)
plot_axis.set_title('Class Imbalance')
plot_axis.set_xlabel('Class')
plot_axis.set_ylabel('Count')
plot_axis.tick_params(axis='x', rotation=0)

# Output Management
plt.tight_layout()
plt.savefig('../eda_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

# Phase E — Multivariate Analysis 

### 9. Bivariate Analysis

In [ ]:
# Constants
CLASS_LABELS = ['GALAXY', 'QSO', 'STAR']
CLASS_COLOR_PALETTE = ['#4C72B0', '#DD8452', '#55A868']
CLASS_COLOR_MAPPING = dict(zip(CLASS_LABELS, CLASS_COLOR_PALETTE))
RANDOM_SEED = 42
SAMPLE_SIZE = 3000
HEATMAP_COLOR_MAP = 'YlOrRd'
DECIMAL_FORMAT = '.1f'

# Figure Setup
figure, plot_axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Redshift by class
for class_name, class_color in CLASS_COLOR_MAPPING.items():
    class_subset = training_dataset[training_dataset['class'] == class_name]
    redshift_values = class_subset['redshift']
    plot_axes[0, 0].hist(redshift_values, bins=50, alpha=0.6, label=class_name, color=class_color)
plot_axes[0, 0].set_title('Redshift Distribution by Class')
plot_axes[0, 0].set_xlabel('Redshift')
plot_axes[0, 0].legend()

# 2. Alpha vs Delta scatter (sample)
dataset_sample = training_dataset.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)
for class_name in CLASS_LABELS:
    sampled_class_subset = dataset_sample[dataset_sample['class'] == class_name]
    plot_axes[0, 1].scatter(
        sampled_class_subset['alpha'],
        sampled_class_subset['delta'],
        c=CLASS_COLOR_MAPPING[class_name],
        label=class_name,
        alpha=0.4,
        s=5
    )
plot_axes[0, 1].set_title('Sky Position (Alpha vs Delta)')
plot_axes[0, 1].set_xlabel('Alpha (RA)')
plot_axes[0, 1].set_ylabel('Delta (Dec)')
plot_axes[0, 1].legend()

# 3. Spectral type vs Class heatmap
spectral_vs_class_crosstab = pd.crosstab(
    training_dataset['spectral_type'],
    training_dataset['class'],
    normalize='index'
) * 100

sns.heatmap(
    spectral_vs_class_crosstab,
    annot=True,
    fmt=DECIMAL_FORMAT,
    cmap=HEATMAP_COLOR_MAP,
    ax=plot_axes[1, 0]
)
plot_axes[1, 0].set_title('Spectral Type vs Class (%)')
plot_axes[1, 0].set_xlabel('Class')
plot_axes[1, 0].set_ylabel('Spectral Type')

# 4. Galaxy population vs Class heatmap
galaxy_vs_class_crosstab = pd.crosstab(
    training_dataset['galaxy_population'],
    training_dataset['class'],
    normalize='index'
) * 100

sns.heatmap(
    galaxy_vs_class_crosstab,
    annot=True,
    fmt=DECIMAL_FORMAT,
    cmap=HEATMAP_COLOR_MAP,
    ax=plot_axes[1, 1]
)
plot_axes[1, 1].set_title('Galaxy Population vs Class (%)')
plot_axes[1, 1].set_xlabel('Class')
plot_axes[1, 1].set_ylabel('Galaxy Population')

# Output Management
plt.tight_layout()
plt.savefig('../eda_bivariate.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== Redshift Statistics by Class ===")
redshift_summary_by_class = training_dataset.groupby('class')['redshift'].agg(['mean', 'std', 'min', 'max'])
print(redshift_summary_by_class.round(3))

print("\n=== Spectral Type vs Class (Counts) ===")
spectral_vs_class_cross_tab = pd.crosstab(training_dataset['spectral_type'], training_dataset['class'])
print(spectral_vs_class_cross_tab)

print("\n=== Galaxy Population vs Class (Counts) ===")
galaxy_vs_class_cross_tab = pd.crosstab(training_dataset['galaxy_population'], training_dataset['class'])
print(galaxy_vs_class_cross_tab)

### 10. Correlation

In [ ]:
# Constants
NUMERICAL_FEATURES = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']

# Figure Setup
figure, plot_axis = plt.subplots(figsize=(8, 6))
numerical_data_subset = training_dataset[NUMERICAL_FEATURES]
correlation_matrix = numerical_data_subset.corr()
sns.heatmap(
    correlation_matrix,
    ax=plot_axis,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    annot_kws={"size": 8}
)
plot_axis.set_title('Feature Correlation')

# Output Management
plt.tight_layout()
plt.savefig('../eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== Highly Correlated Feature Pairs (|r| > 0.7) ===")
correlation_pairs = correlation_matrix.unstack()
correlation_pairs = correlation_pairs[correlation_pairs.index.get_level_values(0) != correlation_pairs.index.get_level_values(1)]
high_correlation_pairs = correlation_pairs[correlation_pairs.abs() > 0.7].sort_values(ascending=False)
print(high_correlation_pairs)

# Summary

In [ ]:
# Configurations / Formatting Constants
BORDER_LINE = "=" * 60
PERCENTAGE_MULTIPLIER = 100

print(BORDER_LINE)
print("EDA SUMMARY")
print(BORDER_LINE)

# 1. Dataset Size
train_rows_count, train_columns_count = training_dataset.shape
test_rows_count, test_columns_count = testing_dataset.shape

print("\nDataset Size:")
print(f"   Train: {train_rows_count:,} rows × {train_columns_count} columns")
print(f"   Test:  {test_rows_count:,} rows × {test_columns_count} columns")

# 2. Class Distribution
print("\nClass Distribution:")
class_value_counts = training_dataset['class'].value_counts()
total_train_records = len(training_dataset)

for class_label, record_count in class_value_counts.items():
    class_percentage = (record_count / total_train_records) * PERCENTAGE_MULTIPLIER
    print(f"   {class_label:<8}: {record_count:>7,} ({class_percentage:.1f}%)")

# 3. Duplicates & Missing Values
print("\nDuplicate Rows:", training_dataset.duplicated().sum())
total_missing_values = training_dataset.isnull().sum().sum()
print("Missing Values:", total_missing_values if total_missing_values > 0 else "NONE")

# 4. Redshift Statistics by Class
print("\nRedshift Statistics by Class:")
redshift_summary_by_class = training_dataset.groupby('class')['redshift'].agg(['mean', 'std', 'min', 'max'])
print(redshift_summary_by_class.round(3))

# 5. Insights
print("\nKey Insights:")
print("   1. Redshift is the strongest separator (STARs ≈ 0, QSOs > 1)")
print("   2. Color indices (u-g, g-r) differ significantly across classes")
print("   3. Classes are imbalanced → Balanced Accuracy metric used")
print("   4. Spectral type M dominates → mostly galaxies")
print("   5. Sky position (alpha/delta) shows no class separation")